In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path(r"C:\mle-01-p1-team3")
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from retrieve import retrieve

c:\mle-01-p1-team3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3005.97it/s]


### Retriever 반환 형태 확인

In [2]:
question = "표창을 사용하는 직업 모두 정리해봐"

retrieved_docs = retrieve(question, k=5)

print("반환 타입:", type(retrieved_docs))
print("검색 결과 수:", len(retrieved_docs))

if retrieved_docs:
    print("첫 번째 결과 타입:", type(retrieved_docs[0]))
    print("첫 번째 결과:")
    print(retrieved_docs[0])
else:
    print("검색 결과가 없습니다.")

검색 범위: jobs
반환 타입: <class 'list'>
검색 결과 수: 5
첫 번째 결과 타입: <class 'dict'>
첫 번째 결과:
{'rank': 1, 'id': 'chunk_1472', 'page_content': '직업명: 나이트로드\n별명: 그림자 속에 숨은 존재\n설명: 다양한 종류의 표창을 능수능란하게 다루는 도적입니다. 끊임없는 표창 투척과 함께 암살자의 표식, 부적, 두루마리로 더욱 많은 표창을 소환하여 적에게 강력한 피해를 주고 전장을 제압합니다.\n주스탯: LUK (행운)\n무기: 표창, 아대', 'metadata': {'source': 'jobs', 'job_id': '30', 'chunk_index': 1472, 'category': '도적', 'name': '나이트로드', 'url': 'https://maplestory.nexon.com/Guide/N23Job/View/30'}, 'distance': 0.5280954539775848, 'score': 0.47190454602241516}


In [3]:
for doc in retrieved_docs:
    metadata = doc.get("metadata") or {}
    print(
        metadata.get("source"),
        metadata.get("name"),
        f"score={doc.get('score', 0):.4f}",
    )

jobs 나이트로드 score=0.4719
jobs 보우마스터 score=0.4298
jobs 배틀메이지 score=0.3988
jobs 나이트워커 score=0.3951
jobs 아크메이지(썬,콜) score=0.3628


### Retriever 결과를 LLM context 문자열로 변환

In [4]:
def build_context(documents):
    """Retriever 결과(list[dict])를 LLM에 전달할 context 문자열로 변환합니다."""
    if not documents:
        return "관련 검색 결과가 없습니다."

    context_parts = []

    for i, doc in enumerate(documents, start=1):
        metadata = doc.get("metadata") or {}

        source = metadata.get("source", "unknown")
        title = (
            metadata.get("title")
            or metadata.get("name")
            or "제목 없음"
        )
        chunk_id = doc.get("id", "")
        url = metadata.get("url", "")
        page_content = doc.get("page_content", "")

        context_part = f"""
[검색 결과 {i}]
문서 유형: {source}
문서 제목: {title}
chunk_id: {chunk_id}

내용:
{page_content}

출처 URL:
{url}
""".strip()

        context_parts.append(context_part)

    return "\n\n---\n\n".join(context_parts)

In [5]:
question = "표창을 사용하는 직업 모두 정리해봐"

retrieved_docs = retrieve(question, k=5)
context = build_context(retrieved_docs)

print(context)

검색 범위: jobs
[검색 결과 1]
문서 유형: jobs
문서 제목: 나이트로드
chunk_id: chunk_1472

내용:
직업명: 나이트로드
별명: 그림자 속에 숨은 존재
설명: 다양한 종류의 표창을 능수능란하게 다루는 도적입니다. 끊임없는 표창 투척과 함께 암살자의 표식, 부적, 두루마리로 더욱 많은 표창을 소환하여 적에게 강력한 피해를 주고 전장을 제압합니다.
주스탯: LUK (행운)
무기: 표창, 아대

출처 URL:
https://maplestory.nexon.com/Guide/N23Job/View/30

---

[검색 결과 2]
문서 유형: jobs
문서 제목: 보우마스터
chunk_id: chunk_1465

내용:
직업명: 보우마스터
별명: 속사의 정점
설명: 활의 정점에 도달해 다양한 화살로 적을 섬멸하는 궁수입니다. 화살을 연속적으로 발사하는 속사 공격과 상황에 맞춰 기능을 선택할 수 있는 추가 화살을 끊임없이 퍼부어 전장을 뒤덮습니다.
주스탯: DEX (민첩)
무기: 활

출처 URL:
https://maplestory.nexon.com/Guide/N23Job/View/23

---

[검색 결과 3]
문서 유형: jobs
문서 제목: 배틀메이지
chunk_id: chunk_1458

내용:
직업명: 배틀메이지
별명: 최전선의 마법사
설명: 실전에 특화되어 근접전이 가능한 마법사입니다. 높은 기동력으로 접근해 스태프를 휘둘러 공격하며, 어둠의 힘으로 적을 응징하고 다양한 오라로 동료를 지원합니다.
주스탯: INT (지력)
무기: 스태프

출처 URL:
https://maplestory.nexon.com/Guide/N23Job/View/17

---

[검색 결과 4]
문서 유형: jobs
문서 제목: 나이트워커
chunk_id: chunk_1475

내용:
직업명: 나이트워커
별명: 비정한 어둠의 기사
설명: 정령 다크니스의 힘을 받아들여 그림자와 어둠의 힘을 사용하는 도적입니다. 그림자로 빚어낸 배트와 자신의 행동을 따라하는 그림